# 📖 [04] 대규모 문헌 비교 및 연구 공백 분석

이 노트북은 **Research Gap Analyzer** 기능의 핵심 연산인 다단계 LLM 오케스트레이션과 구조화 정보 추출(Structured Output), 그리고 엄격한 학술 번역 체인 설계 방식을 분석하는 독립형 실습 스크립트입니다.

---

## 💡 3분 배경지식: Structured Output & Translation Chain
1. **with_structured_output API**:
   - LLM이 일반 줄글로 답변을 생성하면 파싱 중 예외가 발생하거나 포맷이 어긋날 수 있습니다. Pydantic 스키마 모델을 지정하여 `with_structured_output` API를 가동하면 JSON 스키마 제약을 모델에 걸어주어, 항상 동일한 구조의 객체(DTO)를 안전하게 획득할 수 있습니다.
2. **학술 전문 용어 번역 체인**:
   - 학술 요약문을 기계적으로 자동 번역하면 'Transformer'가 '변환기'로 번역되는 등 심각한 오번역이 생깁니다. 이를 방지하기 위해 특수한 용어 매핑 규칙(Glossary)을 프롬프트 내에 기입하여 번역을 수행합니다.

---

## 🔄 2단계 분석 및 합성 흐름
```mermaid
graph TD
    A[가상 논문 4편 초록] -->|1단계: 개별 분석| B[PaperAnalysisResult 4개]
    B -->|2단계: 종합 합성| C[ResearchGapMatrix 영문 생성]
    C -->|3단계: 번역 체인| D[한글 연구 공백 및 로드맵]
```

### 1. 환경 준비 및 Pydantic DTO 선언

In [ ]:
import sys
import os

sys.path.append(os.path.abspath("../backend"))

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from api.common.config import settings

# 1. 개별 논문 분석 DTO
class PaperAnalysisResult(BaseModel):
    title: str = Field(description="논문 제목")
    problems_solved: list[str] = Field(description="해당 논문이 해결한 핵심 기법이나 문제 2개 이하")
    limitations: list[str] = Field(description="논문 본문에 기술되었거나 유추되는 한계점 2개 이하")

# 2. 종합 연구 공백 매트릭스 DTO
class ResearchGapMatrix(BaseModel):
    common_limitations: list[str] = Field(description="분석 대상 논문들 전반에서 식별되는 공통적 한계점 3개")
    suggested_directions: list[str] = Field(description="이를 극복하기 위해 제안하는 구체적이고 혁신적인 AI 연구 로드맵 3가지")

print("DTO 및 LLM 연동 설정 완료!")

### 2. 가상 학술 논문 4개 정의 (독립형 데이터 자급자족)
데이터베이스 연결 의존성 없이 즉시 LLM 분석을 테스트할 수 있도록 가상의 논문 4개의 데이터를 직접 텍스트로 준비합니다.

In [ ]:
mock_papers = [
    {
        "title": "Paper A: Deep Learning for DNA Sequence Classification",
        "abstract": "We propose a 1D CNN model to classify genomics datasets. The model achieves 95% accuracy but requires large training times and does not capture long-range interactions between nucleotides."
    },
    {
        "title": "Paper B: Transformer Models for Genomics Sequences",
        "abstract": "This study applies self-attention mechanisms to genomics sequences to handle long-range dependencies. However, attention computation complexity scales quadratically, making it unusable for extra-long sequences."
    },
    {
        "title": "Paper C: Linear Attention for Bioinformatics",
        "abstract": "To solve quadratic cost, we implement linear attention mechanisms on DNA sequences. It improves speed but suffers from significant loss in classification accuracy and representation quality."
    },
    {
        "title": "Paper D: Mamba Architecture for Long-Sequence Gene Encoding",
        "abstract": "We apply state space models (Mamba) to genomic sequences. It achieves linear scalability and preserves accuracy, but it is highly sensitive to initialization parameters and lacks open-source pretrained models."
    }
]
print(f"{len(mock_papers)}개의 가상 논문 초록이 바인딩되었습니다.")

### 3. [1단계] 개별 논문 구조화 분석 (Problems & Limitations Extraction)
`with_structured_output` API를 활용하여 각 논문으로부터 해결책과 한계점 정보를 정밀하게 분리해냅니다.

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
structured_llm_individual = llm.with_structured_output(PaperAnalysisResult)

analyzed_results = []
print("논문 개별 분석 연산을 개시합니다...")
for paper in mock_papers:
    prompt = f"""Analyze the following academic paper abstract and extract details according to the schema:
    Title: {paper['title']}
    Abstract: {paper['abstract']}
    """
    result = structured_llm_individual.invoke(prompt)
    # 타입 검증 (타입 가드 적용)
    if not isinstance(result, PaperAnalysisResult):
        raise TypeError("Expected PaperAnalysisResult DTO")
        
    analyzed_results.append(result)
    print(f"✅ '{result.title}' 분석 완료")
    print(f"   - 해결책: {result.problems_solved}")
    print(f"   - 한계점: {result.limitations}\n")

### 4. [2단계] 공통 연구 공백 합성 (Research Gap Matrix Synthesis)
개별 분석 결과를 취합하여, 대상 학술 분야의 핵심 공백과 미래 연구 로드맵을 하나의 구조화 데이터로 합성합니다.

In [ ]:
structured_llm_matrix = llm.with_structured_output(ResearchGapMatrix)

# 분석 데이터 병합
input_context = "\n\n".join([
    f"Title: {r.title}\nProblems Solved: {r.problems_solved}\nLimitations: {r.limitations}"
    for r in analyzed_results
])

synthesis_prompt = f"""Based on the analysis results of these 4 papers, synthesize the collective research gap.
Identify common critical limitations and propose 3 future research roadmap directions to overcome them.

Analysis Results:
{input_context}
"""

matrix_result = structured_llm_matrix.invoke(synthesis_prompt)
if not isinstance(matrix_result, ResearchGapMatrix):
    raise TypeError("Expected ResearchGapMatrix DTO")

print("=== 합성된 연구 공백 매트릭스 (영문) ===")
print(f"1. 공통 한계점: {matrix_result.common_limitations}")
print(f"2. 제안 로드맵: {matrix_result.suggested_directions}")

### 5. [3단계] 전문 학술 한글 번역 체인 (Translation Engine)
학술 전문 영어 용어가 오역되지 않고 깔끔한 격식체 명사형 종결어미(`~함`, `~ 설계`)로 마무리되도록 번역 프롬프트 체인을 실행하여 품질을 비교합니다.

In [ ]:
translation_prompt_template = """
Translate the following academic research gap analysis report into Korean.

### ⚠️ Translation Rules:
1. Preserve standard machine learning and computer science terminology in Korean pronunciation or English acronyms instead of literal translation:
   - Transformer -> 트랜스포머
   - Attention -> 어텐션 메커니즘
   - Linear Attention -> 선형 어텐션
   - State Space Model -> 상태 공간 모델 (SSM)
   - Mamba -> 맘바
2. Write in a formal, concise academic tone. Use noun-based endings (e.g., '~을 해결함', '~ 설계', '~ 연구') for bullet points.
3. Output only the final translated text.

Content to translate:
Common Limitations:
{common_limitations}

Suggested Directions:
{suggested_directions}
"""

final_prompt = translation_prompt_template.format(
    common_limitations="\n- ".join(matrix_result.common_limitations),
    suggested_directions="\n- ".join(matrix_result.suggested_directions)
)

translated_output = llm.invoke(final_prompt).content

print("=== 정밀 한글 번역 결과 ===\n")
print(translated_output)